## **Lab: Transformer for Extractive Question Answering**
---
### What this lab covers
You will build an end-to-end extractive QA system (for Vietnamese) from scratch (no pretrained encoders).
You’ll implement:

* A word-level tokenizer with character offsets

* Sliding window with doc stride for long contexts

* A TransformerEncoder + QA head (start/end logits)

* Exact Match (EM) and F1 evaluation, aggregated per original sample

* Training curves (loss, EM, F1) visualization

**Dataset:** [UIT-ViQuAD 2.0](https://huggingface.co/datasets/taidng/UIT-ViQuAD2.0) via Hugging Face Datasets. We *exclude unanswerable items* and create a test set from 5% of train for reproducible grading.

> **Workspace & Data Preparation (Drive, splits, CSV)**

* Mounts Drive and creates the lab workspace: `MyDrive/Lab 4 - Transformer for QA`.
* Creates subfolders: `data/` and `checkpoints/`.
* Loads `taidng/UIT-ViQuAD2.0`, converts to SQuAD-like rows, and filters out `is_impossible=True`.
* Splits data:

  * test = 5% of train (using original indices to avoid leakage).
  * val = Hugging Face `validation` if present (after filtering). Otherwise take 5% from the remaining train.
* Writes `data/train.csv`, `data/val.csv`, `data/test.csv`.

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

# Create and move into the lab folder on Drive
import os, pathlib, random
LAB_DIR = "/content/gdrive/MyDrive/Lab 4 - Transformer for QA"
pathlib.Path(LAB_DIR).mkdir(parents=True, exist_ok=True)

DATA_DIR = pathlib.Path(LAB_DIR) / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = pathlib.Path(LAB_DIR) / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

%cd "{LAB_DIR}"

# Optional: small README to document generated files
from IPython.display import display
with open("README.txt", "w", encoding="utf-8") as f:
    f.write(
        "Lab 4 - Transformer for QA (UIT-ViQuAD2.0)\n"
        "Data files: train.csv, val.csv, test.csv (filtered is_impossible=False)\n"
        "test is 5% of train; if no validation split, val = 5% of remaining train.\n"
    )

# Install deps for data prep
!pip -q install datasets tqdm

# Load dataset
from datasets import load_dataset
import pandas as pd

HF_NAME = "taidng/UIT-ViQuAD2.0"
ds = load_dataset(HF_NAME)

def split_to_df(dset):
    """
    Convert a HF split to a SQuAD-like DataFrame and filter out unanswerable (is_impossible=True).
    Keeps only the first gold answer (text + answer_start).
    """
    rows = []
    for ex in dset:
        # Skip unanswerable items
        if bool(ex.get("is_impossible", False)):
            continue
        answers = ex.get("answers", {}) or {}
        texts = answers.get("text", []) or []
        starts = answers.get("answer_start", []) or []
        if len(texts) == 0:
            # Extra safety: drop items without answers
            continue

        atext = str(texts[0])
        astart = int(starts[0]) if len(starts) > 0 else 0

        rows.append({
            "id": str(ex.get("id", ex.get("uit_id", ""))),
            "title": str(ex.get("title", "")),
            "context": str(ex["context"]),
            "question": str(ex["question"]),
            "answer": atext,
            "answer_start": astart,
        })
    return pd.DataFrame(rows)

# Build train DataFrame (always present)
train_df_full = split_to_df(ds["train"])

# --- TEST = 5% of TRAIN (correct: drop by original indices, not by reset indices) ---
SEED = 42
test_frac = 0.05
_test_sample = train_df_full.sample(frac=test_frac, random_state=SEED)  # original indices preserved
test_df = _test_sample.reset_index(drop=True)
train_rest = train_df_full.drop(_test_sample.index).reset_index(drop=True)

# --- VALIDATION ---
if "validation" in ds:
    val_df = split_to_df(ds["validation"])
    # If validation becomes too small after filtering, fallback to slicing from train_rest
    if len(val_df) < 100:
        _aux_val = train_rest.sample(frac=test_frac, random_state=SEED+1)
        val_df = _aux_val.reset_index(drop=True)
        train_rest = train_rest.drop(_aux_val.index).reset_index(drop=True)
else:
    # No validation split → take 5% from remaining train
    _val_sample = train_rest.sample(frac=test_frac, random_state=SEED+1)
    val_df = _val_sample.reset_index(drop=True)
    train_rest = train_rest.drop(_val_sample.index).reset_index(drop=True)

# Save CSVs
train_path = pathlib.Path(LAB_DIR) / "data/train.csv"
val_path   = pathlib.Path(LAB_DIR) / "data/val.csv"
test_path  = pathlib.Path(LAB_DIR) / "data/test.csv"

train_rest.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Saved:\n- train.csv: {len(train_rest)} rows"
      f"\n- val.csv:   {len(val_df)} rows"
      f"\n- test.csv:  {len(test_df)} rows")

Mounted at /content/gdrive
/content/gdrive/MyDrive/Lab 4 - Transformer for QA


README.md:   0%|          | 0.00/6.04k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 5.20MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  735kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (2 outputs left out)

Generating train split:   0%|          | 0/28454 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3814 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7301 [00:00<?, ? examples/s]

Saved:
- train.csv: 18276 rows
- val.csv:   2653 rows
- test.csv:  962 rows


> **Environment check:**
Confirms PyTorch and GPU availability.

In [2]:
!pip -q install tqdm

import torch, platform
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Python:", platform.python_version())

PyTorch: 2.11.0+cu130 | CUDA: 13.0 | GPU: Tesla T4
Python: 3.13.15


> **Config:** Centralizes all paths and hyperparameters for easy tweaks.

In [3]:
from dataclasses import dataclass

@dataclass
class CFG:
    # Data
    train_path: str = "data/train.csv"
    val_path:   str = "data/val.csv"
    test_path:  str = "data/test.csv"

    # Sequence & batching
    max_len:    int = 512
    doc_stride: int = 96
    batch_size: int = 64
    epochs:     int = 3

    # Optimization
    lr:         float = 3e-4
    wd:         float = 0.01

    # Transformer
    d_model:    int = 512
    n_heads:    int = 8
    n_layers:   int = 6
    d_ff:       int = 2048
    dropout:    float = 0.1

    # Other
    seed:       int = 42
    save_path:  str = "checkpoints/vanilla_transformer_viquad.pt"

cfg = CFG()
cfg

CFG(train_path='data/train.csv', val_path='data/val.csv', test_path='data/test.csv', max_len=512, doc_stride=96, batch_size=64, epochs=3, lr=0.0003, wd=0.01, d_model=512, n_heads=8, n_layers=6, d_ff=2048, dropout=0.1, seed=42, save_path='checkpoints/vanilla_transformer_viquad.pt')

> **Seeding & device:**
Sets deterministic seeds and chooses GPU if present.

In [4]:
import random, os, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(cfg.seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

> **Load data splits:** Reads `train.csv`, `val.csv`, and `test.csv` produced in Cell 0.

In [5]:
import pandas as pd

train_df = pd.read_csv(cfg.train_path)
val_df   = pd.read_csv(cfg.val_path)
test_df  = pd.read_csv(cfg.test_path)

print("Sizes:", len(train_df), len(val_df), len(test_df))

# Quick peek
display(train_df.head(2))
display(val_df.head(2))
display(test_df.head(2))

Sizes: 18276 2653 962


,id,title,context,question,answer,answer_start
0,0001-0001-0001,Phạm Văn Đồng,Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4...,Tên gọi nào được Phạm Văn Đồng sử dụng khi làm...,Lâm Bá Kiệt,507
1,0001-0001-0002,Phạm Văn Đồng,Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4...,Phạm Văn Đồng giữ chức vụ gì trong bộ máy Nhà ...,Thủ tướng,60


,id,title,context,question,answer,answer_start
0,0001-0001-0001,Paris,Paris nằm ở điểm gặp nhau của các hành trình t...,Paris đạt được thành quả gì sau khoảng 4 thế k...,trở thành một trong những trung tâm văn hóa củ...,565
1,0001-0001-0003,Paris,Paris nằm ở điểm gặp nhau của các hành trình t...,Kinh tế xung quanh kinh đô ánh sáng mạnh về gì?,giáo dục và nghệ thuật,323


,id,title,context,question,answer,answer_start
0,0008-0008-0004,Albert Einstein,"Trong cùng năm, Mileva Marić, người vợ tương l...",Tại sao Marić lại trượt bài thi?,có điểm kém trong chuyên đề Lý thuyết hàm,406
1,0070-0005-0003,Birmingham,Quá trình Birmingham phát triển thành một trun...,"Vào thế kỉ XIV, Birmingham trở thành thị trấn ...",lớn thứ ba,733


> **Tokenizer (word-level) & Vocabulary (with offsets):**
Builds a simple regex tokenizer that supports Vietnamese diacritics, returns token offsets for char → token span alignment, and creates a vocabulary.

In [6]:
import re
from collections import Counter

SPECIAL_TOKENS = ["[PAD]","[CLS]","[SEP]","[UNK]"]
PAD, CLS, SEP, UNK = 0,1,2,3

# Regex: words (letters/digits incl. Vietnamese range) or single non-space symbols
WORD_RE = re.compile(r"[A-Za-zÀ-ỹ0-9_]+|[^\sA-Za-zÀ-ỹ0-9_]")

def tokenize_with_offsets(text):
    """
    Tokenize text and return:
      - tokens: list[str]
      - offsets: list[(start_char, end_char)] per token
    Offsets are used later to map char spans to token indices.
    """
    tokens, offsets = [], []
    for m in WORD_RE.finditer(text):
        s,e = m.span()
        tokens.append(m.group())
        offsets.append((s,e))
    return tokens, offsets

def build_vocab(df_list, min_freq=1, max_size=80000):
    """
    Build a word-level vocabulary from question + context text.
    Adds special tokens at fixed indices.
    """
    counter = Counter()
    for df in df_list:
        for col in ["question","context"]:
            for x in df[col].astype(str).tolist():
                toks,_ = tokenize_with_offsets(x.lower())
                counter.update(toks)
    vocab = {tok:i for i,tok in enumerate(SPECIAL_TOKENS)}
    for tok, freq in counter.most_common():
        if tok in vocab: continue
        if freq < min_freq: break
        if len(vocab) >= max_size: break
        vocab[tok] = len(vocab)
    ivocab = {i:t for t,i in vocab.items()}
    return vocab, ivocab

vocab, ivocab = build_vocab([train_df, val_df])
len(vocab)

19285

> **Encode (question, context) with doc stride**

* Creates windows of the form `[CLS] question [SEP] context_window [SEP]`.

* If context is too long, it slides by `doc_stride`. Stores `offsets` and `sequence_ids` to locate the context area later.

In [7]:
def tokens_to_ids(tokens, vocab):
    return [vocab.get(tok.lower(), UNK) for tok in tokens]

def encode_pair(sample_id, question, context, max_len, doc_stride):
    """
    Return a list of windows. Each window contains:
      - input_ids, token_type_ids, attention_mask
      - offsets (for span alignment) and sequence_ids (0=question, 1=context)
      - sample_id (original example key)
    """
    q_toks, q_offs = tokenize_with_offsets(question)
    c_toks, c_offs = tokenize_with_offsets(context)
    q_ids = tokens_to_ids(q_toks, vocab)
    c_ids = tokens_to_ids(c_toks, vocab)

    # Reserved tokens: [CLS] + question + [SEP] + context + [SEP]
    head_len = 1 + len(q_ids) + 1
    max_c_len = max_len - head_len - 1  # remaining space for context + final [SEP]
    if max_c_len <= 0:
        # If question is too long, truncate the question first
        q_ids = q_ids[:max_len-3]
        q_toks = q_toks[:max_len-3]
        q_offs = q_offs[:max_len-3]
        head_len = 1 + len(q_ids) + 1
        max_c_len = max_len - head_len - 1

    windows = []
    start = 0
    while True:
        c_slice_ids  = c_ids[start:start+max_c_len]
        c_slice_offs = c_offs[start:start+max_c_len]

        input_ids = [CLS] + q_ids + [SEP] + c_slice_ids + [SEP]
        token_type_ids = [0]*(1+len(q_ids)+1) + [1]*(len(c_slice_ids)+1)
        attention_mask = [1]*len(input_ids)

        # Offsets aligned to tokens; (-1,-1) for specials/PAD
        offsets = [(-1,-1)] + q_offs + [(-1,-1)] + c_slice_offs + [(-1,-1)]
        sequence_ids = [0]*(1+len(q_ids)+1) + [1]*(len(c_slice_ids)+1)

        # Pad to max_len
        if len(input_ids) < max_len:
            pad_len = max_len - len(input_ids)
            input_ids      += [PAD]*pad_len
            token_type_ids += [0]*pad_len
            attention_mask += [0]*pad_len
            offsets        += [(-1,-1)]*pad_len
            sequence_ids   += [-1]*pad_len

        windows.append({
            "sample_id": sample_id,
            "input_ids": input_ids,
            "token_type_ids": token_type_ids,
            "attention_mask": attention_mask,
            "offsets": offsets,
            "sequence_ids": sequence_ids,
        })

        if doc_stride <= 0 or start + max_c_len >= len(c_ids):
            break
        start += doc_stride
    return windows


> **Span alignment (char-to-token)**
* Maps the gold character span `(answer_start, answer_end)` to token indices within a specific window.
* If the answer doesn’t fit the window, we use `(0,0)` (pointing to `[CLS]`) — a common “no-answer in this window” label convention.

In [8]:
def align_labels_for_window(win, answer_text, answer_start_char):
    """
    Map character-level answer span to token indices inside 'win'.
    Returns (start_tok, end_tok). If the span is outside the window, returns (0,0).
    """
    if not answer_text or answer_text.strip()=="":
        return 0, 0
    answer_end_char = answer_start_char + len(answer_text)

    seq_ids = win["sequence_ids"]; offsets = win["offsets"]
    # Locate the context token range in this window
    try:
        ctx_start = next(i for i,s in enumerate(seq_ids) if s == 1)
    except StopIteration:
        return 0, 0
    ctx_end = len(seq_ids) - 1
    while ctx_end >= 0 and seq_ids[ctx_end] != 1:
        ctx_end -= 1
    if ctx_end < ctx_start:
        return 0, 0

    start_tok = end_tok = None
    for idx in range(ctx_start, ctx_end+1):
        s,e = offsets[idx]
        if s == -1:      # special/pad token
            continue
        # Token contains the start char
        if start_tok is None and s <= answer_start_char < e:
            start_tok = idx
        # Token contains the end char
        if e >= answer_end_char and s < answer_end_char:
            end_tok = idx
            break

    if start_tok is None or end_tok is None:
        return 0, 0
    return start_tok, end_tok


> **Dataset & DataLoaders:**
* Creates a dataset of windows per original sample and builds DataLoaders.
* Keeps `sample_id` for merging window predictions later.

In [9]:
import torch
from torch.utils.data import Dataset, DataLoader

class QADataset(Dataset):
    def __init__(self, df, max_len, doc_stride):
        self.rows = []
        for _, r in df.iterrows():
            sid = str(r["id"])
            q = str(r["question"]); c = str(r["context"])
            a = str(r.get("answer","")); s = int(r.get("answer_start",0))

            # Create windows for this (q, c)
            wins = encode_pair(sid, q, c, max_len, doc_stride)
            for w in wins:
                st, ed = align_labels_for_window(w, a, s)
                self.rows.append({
                    "sample_id": sid,
                    "input_ids": torch.tensor(w["input_ids"], dtype=torch.long),
                    "token_type_ids": torch.tensor(w["token_type_ids"], dtype=torch.long),
                    "attention_mask": torch.tensor(w["attention_mask"], dtype=torch.long),
                    "start_positions": torch.tensor(st, dtype=torch.long),
                    "end_positions": torch.tensor(ed, dtype=torch.long),
                })

    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

train_ds = QADataset(train_df, cfg.max_len, cfg.doc_stride)
val_ds   = QADataset(val_df,   cfg.max_len, cfg.doc_stride)
test_ds  = QADataset(test_df,  cfg.max_len, cfg.doc_stride)

train_dl = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                      num_workers=0, pin_memory=False)
val_dl   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False,
                      num_workers=0, pin_memory=False)
test_dl  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False,
                      num_workers=0, pin_memory=False)

len(train_ds), len(val_ds), len(test_ds)


(18547, 2678, 972)

> **Model: Positional Encoding + TransformerEncoder + QA Head:**
Implements a vanilla *Transformer* encoder with sinusoidal positional encoding and a linear head producing start and end logits for each token.

In [10]:
import math, torch.nn as nn

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=4096):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # [1, L, d_model]

    def forward(self, x):  # x: [B, T, d_model]
        # Add positional signal to token embeddings
        return x + self.pe[:, :x.size(1), :]

class VanillaTransformerQA(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, dropout):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.pos = SinusoidalPositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.dropout = nn.Dropout(dropout)
        self.qa_head = nn.Linear(d_model, 2)  # 2 logits/token: start, end

    def forward(self, input_ids, attention_mask, token_type_ids=None,
                start_positions=None, end_positions=None):
        x = self.emb(input_ids)               # [B, T, d_model]
        x = self.pos(x)                       # add positional encoding

        # True for PAD positions → tell Transformer to ignore them
        pad_mask = (attention_mask == 0)      # [B, T]
        x = self.encoder(x, src_key_padding_mask=pad_mask)

        x = self.dropout(x)
        logits = self.qa_head(x)              # [B, T, 2]
        start_logits, end_logits = logits.split(1, dim=-1)
        start_logits = start_logits.squeeze(-1)  # [B, T]
        end_logits   = end_logits.squeeze(-1)    # [B, T]

        loss = None
        if start_positions is not None and end_positions is not None:
            # Standard extractive QA loss: CE(start) + CE(end)
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(start_logits, start_positions) + loss_fn(end_logits, end_positions)
        return {"loss": loss, "start_logits": start_logits, "end_logits": end_logits}


> **Initialize model & optimizer:**
Creates the network and the `AdamW` optimizer.

In [14]:
from tqdm.auto import tqdm
import torch

model = VanillaTransformerQA(
    vocab_size=len(vocab),
    d_model=cfg.d_model, n_heads=cfg.n_heads,
    n_layers=cfg.n_layers, d_ff=cfg.d_ff, dropout=cfg.dropout
).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)


> **Training loop (one epoch):** Runs a single epoch and returns average loss (used for plotting).

In [11]:
def train_one_epoch():
    model.train()
    total = 0.0
    for batch in tqdm(train_dl, desc="Train"):
        # Move tensors to device; leave Python objects (e.g., sample_id) as-is
        batch = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k,v in batch.items()}
        out = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            token_type_ids=batch["token_type_ids"],
            start_positions=batch["start_positions"],
            end_positions=batch["end_positions"],
        )
        loss = out["loss"]
        loss.backward()
        # Gradient clipping improves stability on longer sequences
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); opt.zero_grad()
        total += loss.item()
    return total / max(1, len(train_dl))

> **Decoding & Evaluation (EM/F1 per original sample):**
Aggregates all windows for the same `sample_id` and keeps the best span by sum of log-probs of start+end. Then computes EM and F1 vs. the gold answers.

In [12]:
import string, torch.nn.functional as F

def normalize_text(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = " ".join(s.split())
    return s

def ids_to_tokens(ids, ivocab):
    return [ivocab.get(i, "[UNK]") for i in ids]

def decode_span_from_logits(start_logits, end_logits, input_ids, ivocab, max_len=30):
    # Greedy argmax decoding; bound the span length for robustness
    s = int(torch.argmax(start_logits)); e = int(torch.argmax(end_logits))
    if e < s: e = s
    if e - s + 1 > max_len: e = s + max_len - 1
    toks = ids_to_tokens(input_ids[s:e+1], ivocab)
    toks = [t for t in toks if t not in ["[PAD]","[CLS]","[SEP]"]]
    return " ".join(toks).strip(), s, e

@torch.no_grad()
def evaluate(df_gold, dl):
    model.eval()
    # Map from sample_id → gold text (first answer)
    gold = {str(r["id"]): str(r["answer"]) for _, r in df_gold.iterrows()}
    best_pred = {}   # sample_id -> (pred_text, score)

    for batch in tqdm(dl, desc="Eval"):
        input_ids = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)
        tt   = batch["token_type_ids"].to(device)

        out = model(input_ids=input_ids, attention_mask=attn, token_type_ids=tt)
        s_logits, e_logits = out["start_logits"], out["end_logits"]
        s_logp = F.log_softmax(s_logits, dim=-1)
        e_logp = F.log_softmax(e_logits, dim=-1)

        # Accumulate the best (highest start+end log-prob) span per original sample
        for b in range(input_ids.size(0)):
            sid = batch["sample_id"][b]
            pred, si, ei = decode_span_from_logits(s_logits[b], e_logits[b], input_ids[b].tolist(), ivocab)
            score = s_logp[b, si].item() + e_logp[b, ei].item()
            if (sid not in best_pred) or (score > best_pred[sid][1]):
                best_pred[sid] = (pred, score)

    # Compute EM / F1
    def f1(p, g):
        pt = normalize_text(p).split()
        gt = normalize_text(g).split()
        if len(pt)==0 or len(gt)==0:
            return 1.0 if pt==gt else 0.0
        common = set(pt) & set(gt)
        if not common: return 0.0
        prec = len(common)/len(pt); rec = len(common)/len(gt)
        return 2*prec*rec/(prec+rec)

    n=0; em_sum=0.0; f1_sum=0.0
    for sid, (pred, _) in best_pred.items():
        gold_ans = gold.get(sid, "")
        em_sum += 1.0 if normalize_text(pred) == normalize_text(gold_ans) else 0.0
        f1_sum += f1(pred, gold_ans)
        n += 1
    return (em_sum/n if n else 0.0), (f1_sum/n if n else 0.0)

> **Train loop (multi-epoch) + History + Save best:**
Trains for several epochs, collects loss/EM/F1 into `history`, and saves the best model by validation F1.

In [ ]:
history = {"epoch": [], "train_loss": [], "val_em": [], "val_f1": []}

best_f1 = -1.0
for ep in range(1, cfg.epochs+1):
    tr_loss = train_one_epoch()
    em, f1 = evaluate(val_df, val_dl)

    history["epoch"].append(ep)
    history["train_loss"].append(tr_loss)
    history["val_em"].append(em)
    history["val_f1"].append(f1)

    print(f"[Epoch {ep}] loss={tr_loss:.4f} | EM(val)={em:.4f} | F1(val)={f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), cfg.save_path)
        print(">> Saved best to:", cfg.save_path)

print("Best F1(val):", best_f1)

Train:   0%|          | 0/290 [00:00<?, ?it/s]

> **Plots: Loss & Metrics:**
Visualizes training dynamics — Loss, EM, and F1 across epochs.

In [ ]:
import matplotlib.pyplot as plt

# Training loss
plt.figure()
plt.plot(history["epoch"], history["train_loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss per Epoch")
plt.grid(True)
plt.show()

# Validation EM
plt.figure()
plt.plot(history["epoch"], history["val_em"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Validation EM")
plt.title("Validation EM per Epoch")
plt.grid(True)
plt.show()

# Validation F1
plt.figure()
plt.plot(history["epoch"], history["val_f1"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Validation F1")
plt.title("Validation F1 per Epoch")
plt.grid(True)
plt.show()

> **Final test evaluation:**
Loads the best checkpoint and evaluates on the held-out test split (5% of train).

In [ ]:
# Load best and evaluate on test
model.load_state_dict(torch.load(cfg.save_path, map_location=device))
test_em, test_f1 = evaluate(test_df, test_dl)
print(f"[TEST] EM={test_em:.4f} | F1={test_f1:.4f}")

> **Quick QA Demo — Inference mirror of evaluation**
* Loads the best checkpoint and answers a custom `(question, context)` pair using the same logic as evaluation:
  * Context-only masking.
  * Joint best-span decoding with `end ≥ start` and a max length cap.
  * If the context is long, decodes over all sliding windows and returns the best span.

In [ ]:
# Ensure we're using the best weights
model.load_state_dict(torch.load(cfg.save_path, map_location=device))
model.eval();

import torch.nn.functional as F

def answer(question, context, max_len=None, doc_stride=None):
    """
    Simple inference:
    - Build sliding windows for (question, context)
    - Score each candidate span by start+end log-prob
    - Return the best answer text
    """
    if max_len is None: max_len = cfg.max_len
    if doc_stride is None: doc_stride = cfg.doc_stride

    # Windows: [CLS] question [SEP] context_window [SEP]
    wins = encode_pair("demo", question, context, max_len, doc_stride)

    best_text, best_score = "", -1e9
    for w in wins:
        # Prepare tensors for a single window
        inp = torch.tensor(w["input_ids"]).unsqueeze(0).to(device)
        att = torch.tensor(w["attention_mask"]).unsqueeze(0).to(device)
        tt  = torch.tensor(w["token_type_ids"]).unsqueeze(0).to(device)

        # Forward
        out = model(inp, att, tt)
        s_logits, e_logits = out["start_logits"][0], out["end_logits"][0]

        # Greedy decode + length cap (done inside helper)
        pred, si, ei = decode_span_from_logits(s_logits, e_logits, inp[0].tolist(), ivocab)

        # Score span by log-prob(start) + log-prob(end)
        score = F.log_softmax(s_logits, dim=-1)[si].item() + F.log_softmax(e_logits, dim=-1)[ei].item()
        if score > best_score:
            best_score, best_text = score, pred

    return best_text

# ==== Put your own context & question here ====
context = (
    "Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4 năm 2000) là Thủ tướng đầu tiên của "
    "nước Cộng hòa Xã hội chủ nghĩa Việt Nam từ năm 1976 (từ năm 1981 gọi là Chủ tịch Hội đồng Bộ trưởng) "
    "cho đến khi nghỉ hưu năm 1987. Trước đó ông từng giữ chức vụ Thủ tướng Chính phủ Việt Nam Dân chủ Cộng hòa "
    "từ năm 1955 đến năm 1976. Ông là vị Thủ tướng Việt Nam tại vị lâu nhất (1955–1987). Ông là học trò, cộng sự của "
    "Chủ tịch Hồ Chí Minh. Ông có tên gọi thân mật là Tô, đây từng là bí danh của ông. "
    "Ông còn có tên gọi là Lâm Bá Kiệt khi làm Phó chủ nhiệm cơ quan Biện sự xứ tại Quế Lâm (Chủ nhiệm là Hồ Học Lãm)."
)

question = "Tên gọi nào được Phạm Văn Đồng sử dụng khi làm Phó chủ nhiệm cơ quan Biện sự xứ tại Quế Lâm?"

# ==== Run demo ====
pred = answer(question, context)
print("Q:", question)
print("A:", pred)
